# 05. EfficientNetV2-S Fine-tuning

**입력**: `data/split/` (04_split_dataset.ipynb 실행 완료 필요)

**구성**:
- **Phase 1** (기본 실행): backbone freeze → classifier head만 학습
- **Phase 2** (선택): 마지막 2 block unfreeze → fine-tuning

**체크포인트**: `val_loss` 기준 best 모델 자동 저장

**실행 방법**:
1. Phase 1 셀을 순서대로 실행
2. Phase 2는 Phase 1 수렴 확인 후 선택적으로 실행
3. 최종 테스트 평가는 맨 아래 셀에서 단 1회만 실행

> ⚠️ **torch 미설치 시**: 아래 명령어로 먼저 설치하세요.  
> GPU (CUDA 12.x): `pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121`  
> CPU only: `pip install torch torchvision`

## 클래스 정의

| 세부 라벨 | 설명 | 서비스 대분류 |
|-----------|------|-------------|
| `refrigerator` | 가정용 냉장고 | `refrigerator` |
| `washer_dryer` | 세탁기, 건조기, 세탁건조 일체형 포함 | `washing_drying` |
| `wash_tower` | 상하 결합형 세탁·건조 제품 (Phase 2 추가 예정) | `washing_drying` |

> `washer_dryer`와 `wash_tower`는 서비스에서 모두 `washing_drying`으로 처리됩니다.

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
import config

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, WeightedRandomSampler
    from torchvision import datasets, transforms
    from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
    print(f'PyTorch {torch.__version__}')
except ImportError:
    print('[오류] PyTorch가 설치되어 있지 않습니다.')
    print('GPU (CUDA 12.x): pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121')
    print('CPU only       : pip install torch torchvision')
    raise

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

os.makedirs(config.CHECKPOINTS_DIR, exist_ok=True)

# 클래스 자동 감지 — 하드코딩 없음
# data/split/train/ 하위 폴더 이름 = 클래스 목록
# 새 클래스를 추가해도 이 노트북은 수정할 필요 없음
classes = sorted(
    d for d in os.listdir(os.path.join(config.SPLIT_DIR, 'train'))
    if os.path.isdir(os.path.join(config.SPLIT_DIR, 'train', d))
)
num_classes = len(classes)
print(f'\n클래스 ({num_classes}개): {classes}')

PyTorch 2.11.0+cu128
Device: cuda
GPU   : NVIDIA GeForce RTX 5060 Ti

클래스 (3개): ['refrigerator', 'wash_tower', 'washer_dryer']


In [2]:
# ── Dataset / DataLoader ────────────────────────────────────────────────────────
IMG_SIZE   = 224   # EfficientNetV2-S는 384가 공식이지만 소량 데이터에는 224가 실용적
BATCH_SIZE = 16

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(os.path.join(config.SPLIT_DIR, 'train'), transform=train_transform)
valid_dataset = datasets.ImageFolder(os.path.join(config.SPLIT_DIR, 'valid'), transform=val_transform)
test_dataset  = datasets.ImageFolder(os.path.join(config.SPLIT_DIR, 'test'),  transform=val_transform)

# 클래스 불균형 보정 — WeightedRandomSampler
train_counts   = [len(os.listdir(os.path.join(config.SPLIT_DIR, 'train', c))) for c in classes]
class_weights  = 1.0 / torch.tensor(train_counts, dtype=torch.float)
sample_weights = [class_weights[label] for _, label in train_dataset.imgs]
sampler        = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

# num_workers=0: Windows에서 multiprocessing 오류 방지
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=0, pin_memory=(device.type == 'cuda'))
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=(device.type == 'cuda'))
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=(device.type == 'cuda'))

print(f'Train: {len(train_dataset)}장  |  Valid: {len(valid_dataset)}장  |  Test: {len(test_dataset)}장')
print(f'클래스 매핑: {train_dataset.class_to_idx}')
print(f'Train 클래스별: {dict(Counter(lbl for _, lbl in train_dataset.imgs))}')

Train: 704장  |  Valid: 88장  |  Test: 89장
클래스 매핑: {'refrigerator': 0, 'wash_tower': 1, 'washer_dryer': 2}
Train 클래스별: {0: 201, 1: 230, 2: 273}


In [3]:
# ── 모델 로드 + Phase 1 설정 ────────────────────────────────────────────────────
# EfficientNetV2-S pretrained (ImageNet1K)
# 최초 실행 시 ~85MB 가중치 자동 다운로드
model = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)

# Classifier head 교체 (클래스 수에 맞게)
in_features = model.classifier[1].in_features  # 1280
model.classifier[1] = nn.Linear(in_features, num_classes)

# Phase 1: backbone 전체 freeze → classifier head만 학습
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Phase 1 trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

Phase 1 trainable params: 3,843 / 20,181,331 (0.02%)


In [4]:
# ── 학습 유틸리티 ──────────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * inputs.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * inputs.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += labels.size(0)
    return total_loss / total, correct / total

def plot_history(history, title=''):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history['train_loss'], label='Train')
    ax1.plot(history['val_loss'],   label='Val')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title(f'{title} — Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.plot(history['train_acc'], label='Train')
    ax2.plot(history['val_acc'],   label='Val')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
    ax2.set_title(f'{title} — Accuracy'); ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print('유틸리티 함수 정의 완료')

유틸리티 함수 정의 완료


In [5]:
# ── Phase 1 학습 ────────────────────────────────────────────────────────────────
NUM_EPOCHS_P1 = 15
CKPT_PATH     = os.path.join(config.CHECKPOINTS_DIR, 'best_model.pth')

optimizer_p1 = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3
)
scheduler_p1 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_p1, mode='min', factor=0.5, patience=3
)

history_p1   = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_loss = float('inf')

print('── Phase 1: backbone freeze, classifier head만 학습 ──\n')
for epoch in range(1, NUM_EPOCHS_P1 + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer_p1)
    vl_loss, vl_acc = evaluate(model, valid_loader)
    scheduler_p1.step(vl_loss)

    history_p1['train_loss'].append(tr_loss)
    history_p1['train_acc'].append(tr_acc)
    history_p1['val_loss'].append(vl_loss)
    history_p1['val_acc'].append(vl_acc)

    marker = ''
    if vl_loss < best_val_loss:  # val_loss 기준으로 best 저장
        best_val_loss = vl_loss
        torch.save({
            'epoch': epoch,
            'phase': 'phase1',
            'model_state_dict': model.state_dict(),
            'val_loss': vl_loss,
            'val_acc':  vl_acc,
            'classes':  classes,
        }, CKPT_PATH)
        marker = '  <- best'

    print(f'Epoch {epoch:2d}/{NUM_EPOCHS_P1} | '
          f'Train Loss {tr_loss:.4f} Acc {tr_acc:.3f} | '
          f'Val Loss {vl_loss:.4f} Acc {vl_acc:.3f}{marker}')

print(f'\nPhase 1 완료. Best val_loss: {best_val_loss:.4f}')

── Phase 1: backbone freeze, classifier head만 학습 ──



Epoch  1/15 | Train Loss 0.7863 Acc 0.726 | Val Loss 0.6457 Acc 0.841  <- best


Epoch  2/15 | Train Loss 0.4992 Acc 0.837 | Val Loss 0.4729 Acc 0.886  <- best


Epoch  3/15 | Train Loss 0.4302 Acc 0.852 | Val Loss 0.4199 Acc 0.898  <- best


Epoch  4/15 | Train Loss 0.4038 Acc 0.862 | Val Loss 0.3565 Acc 0.932  <- best


...

Epoch 11/15 | Train Loss 0.2892 Acc 0.903 | Val Loss 0.2939 Acc 0.909


Epoch 12/15 | Train Loss 0.3207 Acc 0.896 | Val Loss 0.2777 Acc 0.909


Epoch 13/15 | Train Loss 0.2616 Acc 0.913 | Val Loss 0.2485 Acc 0.909  <- best


Epoch 14/15 | Train Loss 0.3222 Acc 0.888 | Val Loss 0.2722 Acc 0.920


Epoch 15/15 | Train Loss 0.2718 Acc 0.898 | Val Loss 0.2767 Acc 0.909

Phase 1 완료. Best val_loss: 0.2485


In [6]:
# ── Phase 1 결과 확인 ───────────────────────────────────────────────────────────
plot_history(history_p1, title='Phase 1')

ckpt = torch.load(CKPT_PATH)
print(f'저장된 best checkpoint')
print(f'  phase    : {ckpt["phase"]}')
print(f'  epoch    : {ckpt["epoch"]}')
print(f'  val_loss : {ckpt["val_loss"]:.4f}')
print(f'  val_acc  : {ckpt["val_acc"]:.4f}')

C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\307604516.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


저장된 best checkpoint
  phase    : phase1
  epoch    : 13
  val_loss : 0.2485
  val_acc  : 0.9091


---
## Phase 2 — 선택 사항 (기본 실행 불필요)

Phase 1의 val_acc가 충분히 수렴하지 않았을 때만 실행하세요.

**unfreeze 대상**: `features.6` (마지막 MBConv block) + `features.7` (head conv) + `classifier`

**주의**: 데이터가 적어 overfitting 위험이 있습니다. train/val loss 차이를 주시하세요.

아래 셀들은 필요할 때만 실행하세요.

In [7]:
# ── Phase 2 준비: Phase 1 best weight 로드 + 마지막 block unfreeze ───────────────
NUM_EPOCHS_P2 = 10

model.load_state_dict(torch.load(CKPT_PATH)['model_state_dict'])

for param in model.parameters():
    param.requires_grad = False

# features.6, features.7, classifier만 unfreeze
for name, param in model.named_parameters():
    if any(key in name for key in ['features.6', 'features.7', 'classifier']):
        param.requires_grad = True

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Phase 2 trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

Phase 2 trainable params: 14,895,915 / 20,181,331 (73.81%)


In [8]:
# ── Phase 2 학습 ────────────────────────────────────────────────────────────────
optimizer_p2 = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4  # pretrained layer → 낮은 lr
)
scheduler_p2 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_p2, mode='min', factor=0.5, patience=3
)

history_p2    = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_loss_p2 = float('inf')

print('── Phase 2: features.6, features.7, classifier unfreeze ──\n')
for epoch in range(1, NUM_EPOCHS_P2 + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer_p2)
    vl_loss, vl_acc = evaluate(model, valid_loader)
    scheduler_p2.step(vl_loss)

    history_p2['train_loss'].append(tr_loss)
    history_p2['train_acc'].append(tr_acc)
    history_p2['val_loss'].append(vl_loss)
    history_p2['val_acc'].append(vl_acc)

    marker = ''
    if vl_loss < best_val_loss_p2:  # val_loss 기준으로 best 저장
        best_val_loss_p2 = vl_loss
        torch.save({
            'epoch': epoch,
            'phase': 'phase2',
            'model_state_dict': model.state_dict(),
            'val_loss': vl_loss,
            'val_acc':  vl_acc,
            'classes':  classes,
        }, CKPT_PATH)
        marker = '  <- best'

    print(f'Epoch {epoch:2d}/{NUM_EPOCHS_P2} | '
          f'Train Loss {tr_loss:.4f} Acc {tr_acc:.3f} | '
          f'Val Loss {vl_loss:.4f} Acc {vl_acc:.3f}{marker}')

print(f'\nPhase 2 완료. Best val_loss: {best_val_loss_p2:.4f}')

── Phase 2: features.6, features.7, classifier unfreeze ──



Epoch  1/10 | Train Loss 0.2739 Acc 0.905 | Val Loss 0.1494 Acc 0.932  <- best


Epoch  2/10 | Train Loss 0.1734 Acc 0.940 | Val Loss 0.1511 Acc 0.966


Epoch  3/10 | Train Loss 0.2024 Acc 0.940 | Val Loss 0.1772 Acc 0.932


Epoch  4/10 | Train Loss 0.1746 Acc 0.940 | Val Loss 0.1371 Acc 0.943  <- best


...

Epoch  6/10 | Train Loss 0.1018 Acc 0.969 | Val Loss 0.1331 Acc 0.932


Epoch  7/10 | Train Loss 0.1171 Acc 0.963 | Val Loss 0.1357 Acc 0.955


Epoch  8/10 | Train Loss 0.0976 Acc 0.967 | Val Loss 0.1457 Acc 0.943


Epoch  9/10 | Train Loss 0.0918 Acc 0.956 | Val Loss 0.1029 Acc 0.977  <- best


Epoch 10/10 | Train Loss 0.1066 Acc 0.963 | Val Loss 0.1051 Acc 0.966

Phase 2 완료. Best val_loss: 0.1029


In [9]:
# ── Phase 2 결과 확인 ───────────────────────────────────────────────────────────
plot_history(history_p2, title='Phase 2')

ckpt = torch.load(CKPT_PATH)
print(f'저장된 best checkpoint (갱신 후)')
print(f'  phase    : {ckpt["phase"]}')
print(f'  epoch    : {ckpt["epoch"]}')
print(f'  val_loss : {ckpt["val_loss"]:.4f}')
print(f'  val_acc  : {ckpt["val_acc"]:.4f}')

C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\307604516.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


저장된 best checkpoint (갱신 후)
  phase    : phase2
  epoch    : 9
  val_loss : 0.1029
  val_acc  : 0.9773


---
## 최종 테스트 평가 — 학습 완료 후 단 1회만 실행

> **주의**: 아래 셀은 Phase 1 (또는 Phase 2)이 완전히 끝난 후에만 실행하세요.  
> Test set을 여러 번 보면 그 자체로 data leakage가 됩니다.

> ⚠️ **신뢰도 주의**: 현재 test set은 약 40장으로 매우 적습니다.  
> 정확도 1% 변화가 약 0.4장 차이에 불과하므로,  
> 이 수치는 절대 지표가 아닌 **참고 지표**로만 활용하세요.

In [10]:
# ── 최종 테스트 평가 (best checkpoint, 단 1회) ─────────────────────────────────
ckpt = torch.load(CKPT_PATH)
model.load_state_dict(ckpt['model_state_dict'])

test_loss, test_acc = evaluate(model, test_loader)

print('=' * 50)
print(f'  Checkpoint : Phase {ckpt["phase"]}, Epoch {ckpt["epoch"]}')
print(f'  Test Loss  : {test_loss:.4f}')
print(f'  Test Acc   : {test_acc:.4f}  ({test_acc*100:.1f}%)')
print('=' * 50)
print()
# 현재 test set은 약 40장으로 통계적 신뢰도가 낮습니다.
# 정확도 5% 차이 = 약 2장 차이에 불과합니다.
# 데이터를 더 수집한 후 재평가를 권장합니다.
n_test = len(test_dataset)
print(f'[주의] test set 크기: {n_test}장')
print(f'       정확도 1% = {n_test * 0.01:.1f}장 차이 — 참고 지표로만 활용하세요.')

  Checkpoint : Phase phase2, Epoch 9
  Test Loss  : 0.1856
  Test Acc   : 0.9438  (94.4%)

[주의] test set 크기: 89장
       정확도 1% = 0.9장 차이 — 참고 지표로만 활용하세요.


In [11]:
# ── Confusion Matrix ────────────────────────────────────────────────────────────
all_preds, all_targets = [], []
model.eval()
with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs.to(device))
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_targets.extend(labels.numpy())

n = num_classes
cm = [[0] * n for _ in range(n)]
for pred, true in zip(all_preds, all_targets):
    cm[true][pred] += 1

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im)
ax.set(xticks=range(n), yticks=range(n),
       xticklabels=classes, yticklabels=classes,
       ylabel='실제 클래스', xlabel='예측 클래스',
       title=f'Confusion Matrix (Test {n_test}장)')
max_val = max(max(row) for row in cm)
for i in range(n):
    for j in range(n):
        color = 'white' if cm[i][j] > max_val / 2 else 'black'
        ax.text(j, i, cm[i][j], ha='center', va='center', color=color)
plt.tight_layout()
plt.show()

print('Per-class 정확도:')
for i, cls in enumerate(classes):
    total_cls = sum(cm[i])
    acc_cls   = cm[i][i] / total_cls if total_cls > 0 else 0
    print(f'  {cls}: {cm[i][i]}/{total_cls}  ({acc_cls:.1%})')

Per-class 정확도:
  refrigerator: 25/25  (100.0%)
  wash_tower: 27/29  (93.1%)
  washer_dryer: 32/35  (91.4%)


C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\735248330.py:27: UserWarning: Glyph 50696 (\N{HANGUL SYLLABLE YE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\735248330.py:27: UserWarning: Glyph 52769 (\N{HANGUL SYLLABLE CEUG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\735248330.py:27: UserWarning: Glyph 53364 (\N{HANGUL SYLLABLE KEUL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\735248330.py:27: UserWarning: Glyph 47000 (\N{HANGUL SYLLABLE RAE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\735248330.py:27: UserWarning: Glyph 49828 (\N{HANGUL SYLLABLE SEU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\735248330.py:27: UserWarning: Glyph 49892 (\N{HANGUL SYLLABLE SIL}) missing from fon

---
## 예측 결과 분석

best checkpoint를 사용해 test set 전체의 예측 결과를 수집합니다.

- 오답 이미지 시각화
- 정답이지만 confidence가 낮은 이미지 Top 10
- 정답이면서 confidence가 높은 이미지 Top 10
- 결과 CSV 저장 (`data/metadata/test_predictions.csv`)

In [12]:
# ── 예측 결과 수집 + CSV 저장 ──────────────────────────────────────────────────
import pandas as pd

# best checkpoint 로드
ckpt = torch.load(CKPT_PATH, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

all_results = []
img_idx = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        probs = torch.softmax(model(inputs), dim=1)
        confs, preds = probs.max(dim=1)
        for i in range(inputs.size(0)):
            true_cls = classes[labels[i].item()]
            pred_cls = classes[preds[i].item()]
            all_results.append({
                'image_path': test_dataset.imgs[img_idx][0],
                'true_label': true_cls,
                'pred_label': pred_cls,
                'confidence': round(confs[i].item(), 6),
                'is_correct': true_cls == pred_cls,
            })
            img_idx += 1

result_df = pd.DataFrame(all_results)
csv_path = os.path.join(config.METADATA_DIR, 'test_predictions.csv')
result_df.to_csv(csv_path, index=False, encoding='utf-8-sig')

n_correct = result_df['is_correct'].sum()
n_total = len(result_df)
print(f'예측 수집 완료: {n_total}장  |  정답 {n_correct}장  |  오답 {n_total - n_correct}장')
print(f'CSV 저장: {csv_path}')
result_df.head()

예측 수집 완료: 89장  |  정답 84장  |  오답 5장
CSV 저장: data\metadata\test_predictions.csv


,image_path,true_label,pred_label,confidence,is_correct
0,data\split\test\refrigerator\6a6c4348e1313a0d_...,refrigerator,refrigerator,0.519845,True
1,data\split\test\refrigerator\9e21c7896cb36b7a_...,refrigerator,refrigerator,0.979680,True
2,data\split\test\refrigerator\naver_0025.jpg,refrigerator,refrigerator,0.998960,True
3,data\split\test\refrigerator\naver_0034.jpg,refrigerator,refrigerator,0.999780,True
4,data\split\test\refrigerator\naver_0039.jpg,refrigerator,refrigerator,0.999992,True


In [13]:
# ── 오답 이미지 시각화 ──────────────────────────────────────────────────────────
from PIL import Image as PILImage

wrong_df = result_df[~result_df['is_correct']].reset_index(drop=True)
print(f'오답 {len(wrong_df)}장')

if len(wrong_df) == 0:
    print('오답이 없습니다. 완벽한 예측!')
else:
    n_cols = min(5, len(wrong_df))
    n_rows = (len(wrong_df) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    axes = np.array(axes).reshape(-1) if n_rows * n_cols > 1 else [axes]

    for ax_i, (_, row) in enumerate(wrong_df.iterrows()):
        img = PILImage.open(row['image_path']).convert('RGB')
        axes[ax_i].imshow(img)
        axes[ax_i].set_title(
            f'True: {row["true_label"]}\nPred: {row["pred_label"]}\nConf: {row["confidence"]:.3f}',
            fontsize=9, color='red'
        )
        axes[ax_i].axis('off')

    for ax_i in range(len(wrong_df), len(axes)):
        axes[ax_i].axis('off')

    plt.suptitle(f'오답 이미지 ({len(wrong_df)}장)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

오답 5장


C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\2745919123.py:28: UserWarning: Glyph 50724 (\N{HANGUL SYLLABLE O}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\2745919123.py:28: UserWarning: Glyph 45813 (\N{HANGUL SYLLABLE DAB}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\2745919123.py:28: UserWarning: Glyph 51060 (\N{HANGUL SYLLABLE I}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\2745919123.py:28: UserWarning: Glyph 48120 (\N{HANGUL SYLLABLE MI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\2745919123.py:28: UserWarning: Glyph 51648 (\N{HANGUL SYLLABLE JI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\2745919123.py:28: UserWarning: Glyph 51109 (\N{HANGUL SYLLABLE JANG}) missing from fon

In [14]:
# ── 정답이지만 confidence 낮은 Top 10 ──────────────────────────────────────────
# confidence 색 기준: red < 0.85, orange < 0.95, green >= 0.95
def conf_color(c):
    if c < 0.85:
        return 'red'
    elif c < 0.95:
        return 'orange'
    return 'green'

low_conf_df = (
    result_df[result_df['is_correct']]
    .sort_values('confidence', ascending=True)
    .head(10)
    .reset_index(drop=True)
)
print(f'낮은 confidence 정답 Top 10 (최대):')

n_show = len(low_conf_df)
n_cols = min(5, n_show)
n_rows = (n_show + n_cols - 1) // n_cols if n_show > 0 else 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = np.array(axes).reshape(-1) if n_rows * n_cols > 1 else [axes]

for ax_i, (_, row) in enumerate(low_conf_df.iterrows()):
    img = PILImage.open(row['image_path']).convert('RGB')
    color = conf_color(row['confidence'])
    axes[ax_i].imshow(img)
    axes[ax_i].set_title(
        f'{row["true_label"]}\nConf: {row["confidence"]:.3f}',
        fontsize=9, color=color
    )
    axes[ax_i].axis('off')

for ax_i in range(n_show, len(axes)):
    axes[ax_i].axis('off')

plt.suptitle('정답 & 낮은 confidence Top 10\n(red<0.85  orange<0.95  green≥0.95)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

낮은 confidence 정답 Top 10 (최대):


C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\1555599123.py:39: UserWarning: Glyph 51221 (\N{HANGUL SYLLABLE JEONG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\1555599123.py:39: UserWarning: Glyph 45813 (\N{HANGUL SYLLABLE DAB}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\1555599123.py:39: UserWarning: Glyph 45230 (\N{HANGUL SYLLABLE NAJ}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\1555599123.py:39: UserWarning: Glyph 51008 (\N{HANGUL SYLLABLE EUN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\1555599123.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# ── 정답이면서 confidence 높은 Top 10 ──────────────────────────────────────────
high_conf_df = (
    result_df[result_df['is_correct']]
    .sort_values('confidence', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
print(f'높은 confidence 정답 Top 10 (최대):')

n_show = len(high_conf_df)
n_cols = min(5, n_show)
n_rows = (n_show + n_cols - 1) // n_cols if n_show > 0 else 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = np.array(axes).reshape(-1) if n_rows * n_cols > 1 else [axes]

for ax_i, (_, row) in enumerate(high_conf_df.iterrows()):
    img = PILImage.open(row['image_path']).convert('RGB')
    color = conf_color(row['confidence'])
    axes[ax_i].imshow(img)
    axes[ax_i].set_title(
        f'{row["true_label"]}\nConf: {row["confidence"]:.3f}',
        fontsize=9, color=color
    )
    axes[ax_i].axis('off')

for ax_i in range(n_show, len(axes)):
    axes[ax_i].axis('off')

plt.suptitle('정답 & 높은 confidence Top 10\n(green≥0.95  orange<0.95  red<0.85)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

높은 confidence 정답 Top 10 (최대):


C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\3910148325.py:31: UserWarning: Glyph 51221 (\N{HANGUL SYLLABLE JEONG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\3910148325.py:31: UserWarning: Glyph 45813 (\N{HANGUL SYLLABLE DAB}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\3910148325.py:31: UserWarning: Glyph 45458 (\N{HANGUL SYLLABLE NOP}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\3910148325.py:31: UserWarning: Glyph 51008 (\N{HANGUL SYLLABLE EUN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\3910148325.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
# ── 서비스 대분류 매핑 + CSV 갱신 ──────────────────────────────────────────────
# result_df : Cell 15(예측 수집 셀)에서 생성
# csv_path  : Cell 15에서 정의

result_df['service_true_label'] = result_df['true_label'].map(config.SERVICE_LABEL_MAP)
result_df['service_pred_label'] = result_df['pred_label'].map(config.SERVICE_LABEL_MAP)
result_df['service_is_correct'] = (
    result_df['service_true_label'] == result_df['service_pred_label']
)

result_df.to_csv(csv_path, index=False, encoding='utf-8-sig')

fine_n  = result_df['is_correct'].sum()
svc_n   = result_df['service_is_correct'].sum()
total   = len(result_df)
print(f'세부 라벨 정확도   : {fine_n}/{total}  ({fine_n/total:.1%})')
print(f'서비스 대분류 정확도: {svc_n}/{total}  ({svc_n/total:.1%})')
print(f'CSV 갱신: {csv_path}')
print()

# 세부 오답이지만 서비스 정답인 케이스 (Phase 2 wash_tower 추가 후 의미 있음)
recovered = result_df[~result_df['is_correct'] & result_df['service_is_correct']]
print(f'세부 오답 + 서비스 정답 (wash_tower↔washer_dryer 혼동): {len(recovered)}건')
for _, row in recovered.iterrows():
    print(f'  {row["true_label"]} → {row["pred_label"]}  '
          f'(서비스: {row["service_true_label"]} = {row["service_pred_label"]})  '
          f'conf={row["confidence"]:.3f}')

세부 라벨 정확도   : 84/89  (94.4%)
서비스 대분류 정확도: 89/89  (100.0%)
CSV 갱신: data\metadata\test_predictions.csv

세부 오답 + 서비스 정답 (wash_tower↔washer_dryer 혼동): 5건
  wash_tower → washer_dryer  (서비스: washing_drying = washing_drying)  conf=0.672
  wash_tower → washer_dryer  (서비스: washing_drying = washing_drying)  conf=0.536
  washer_dryer → wash_tower  (서비스: washing_drying = washing_drying)  conf=0.880
  washer_dryer → wash_tower  (서비스: washing_drying = washing_drying)  conf=1.000
  washer_dryer → wash_tower  (서비스: washing_drying = washing_drying)  conf=0.614


In [17]:
# ── 세부 라벨 + 서비스 대분류 Confusion Matrix (나란히) ─────────────────────────
service_classes = sorted(result_df['service_true_label'].unique())
n_svc = len(service_classes)
svc_idx = {c: i for i, c in enumerate(service_classes)}

cm_svc = [[0] * n_svc for _ in range(n_svc)]
for _, row in result_df.iterrows():
    cm_svc[svc_idx[row['service_true_label']]][svc_idx[row['service_pred_label']]] += 1

# 세부 라벨 CM 재계산
n = num_classes
cm_fine = [[0] * n for _ in range(n)]
for _, row in result_df.iterrows():
    cm_fine[classes.index(row['true_label'])][classes.index(row['pred_label'])] += 1

def draw_cm(ax, cm, labels, title, cmap):
    k = len(labels)
    im = ax.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.colorbar(im, ax=ax)
    ax.set(xticks=range(k), yticks=range(k),
           xticklabels=labels, yticklabels=labels,
           ylabel='실제', xlabel='예측', title=title)
    ax.tick_params(axis='x', rotation=20)
    max_v = max(max(r) for r in cm) if any(max(r) for r in cm) else 1
    for i in range(k):
        for j in range(k):
            ax.text(j, i, cm[i][j], ha='center', va='center', fontsize=11,
                    color='white' if cm[i][j] > max_v / 2 else 'black')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
draw_cm(axes[0], cm_fine, classes,         '세부 라벨 CM',     plt.cm.Blues)
draw_cm(axes[1], cm_svc,  service_classes, '서비스 대분류 CM', plt.cm.Greens)
plt.suptitle('세부 라벨 vs 서비스 대분류 Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('서비스 대분류 per-class:')
for i, cls in enumerate(service_classes):
    total_cls = sum(cm_svc[i])
    acc_cls   = cm_svc[i][i] / total_cls if total_cls > 0 else 0
    print(f'  {cls}: {cm_svc[i][i]}/{total_cls}  ({acc_cls:.1%})')

서비스 대분류 per-class:
  refrigerator: 25/25  (100.0%)
  washing_drying: 64/64  (100.0%)


C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\2983621724.py:34: UserWarning: Glyph 50696 (\N{HANGUL SYLLABLE YE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\2983621724.py:34: UserWarning: Glyph 52769 (\N{HANGUL SYLLABLE CEUG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\2983621724.py:34: UserWarning: Glyph 49892 (\N{HANGUL SYLLABLE SIL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\2983621724.py:34: UserWarning: Glyph 51228 (\N{HANGUL SYLLABLE JE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\2983621724.py:34: UserWarning: Glyph 49464 (\N{HANGUL SYLLABLE SE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_15500\2983621724.py:34: UserWarning: Glyph 48512 (\N{HANGUL SYLLABLE BU}) missing from f